# ToM-SLM-QLoRA — Complete Pipeline

End-to-end pipeline for the fine-tuning and evaluation results reported in the accompanying
manuscript: ToMBench loading and splitting, external-benchmark manifest construction, QLoRA
fine-tuning across multiple seeds, a story-level group-split control, training-free few-shot
baselines, an fp16 LoRA control experiment, and integrity checks throughout.

This notebook is a single, self-contained, resumable version of the full pipeline.

## Design notes

- **Item identity.** Every record (ToMBench and all four external benchmarks)
  gets a canonical `item_id` derived from a SHA-256 hash of its normalized
  source, task, story, and question text, plus a `prompt_hash` of the exact
  rendered prompt. Randomized choices (answer-option order, distractor
  sampling) use a hash of the item content as the seed, not Python's built-in
  `hash()`, which is not guaranteed to be stable across processes.
- **Manifests are built once and reused.** External-benchmark item sets are
  written to disk the first time they are constructed and reloaded on every
  subsequent run, so every fine-tuning seed and every few-shot condition is
  evaluated on the exact same items.
- **Group split uses story identity alone.** The story-disjoint control
  groups items by normalized story content only, not by (task, story), so a
  story cannot appear in both train and test merely because it is attached to
  a different task label.
- **Few-shot conditions are kept separate.** A content-bearing "answer-only"
  condition (full task content and gold answer, no reasoning), a content-free
  "synthetic-format-only" condition (unrelated toy questions, output format
  only), and a "rationale-CoT" condition (brief reasoning before the answer)
  are each evaluated with three independent exemplar draws, for both the base
  and fine-tuned model.
- **Resumable by design.** Every expensive step caches results to disk;
  re-running a cell skips computation and reloads from the saved file.

**Runtime (GPU T4/L4/A100 recommended):** ~5–7 h total on a T4 for the full pipeline (base
pipeline ~3–4 h; the group-split, few-shot, and fp16-control experiments add ~2–3 h), less on
an A100.

## Setup

In [ ]:

import sys, os, subprocess, time
import torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_gb:.1f}GB VRAM)')

print('\nInstalling packages (2-3 min)...')
t0 = time.time()
# numpy/pandas/scipy/scikit-learn are left at whatever mutually-compatible
# versions Colab ships with. Pinning older versions here would create an ABI
# mismatch with other preinstalled packages already compiled against the
# newer versions (surfacing as e.g. "numpy.dtype size changed, may indicate
# binary incompatibility"), so this cell only installs packages this
# notebook actually needs that are not already present.
!pip install -q unsloth
!pip install -q "datasets<4.0.0" tqdm
print(f'   {time.time()-t0:.0f}s')
print('Using Colab\'s preinstalled numpy/pandas/scipy/scikit-learn (versions below):')
import numpy, pandas, scipy, sklearn
print(f'   numpy={numpy.__version__} pandas={pandas.__version__} scipy={scipy.__version__} scikit-learn={sklearn.__version__}')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = 'ToMBench_Results'   # must match the folder used by any prior run sharing this Drive
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
OUTPUT_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')
print(f'Output dir: {OUTPUT_DIR}')


## Integrity utilities: stable hashing, canonical item IDs, matched-join validation

In [ ]:

import hashlib, json as _json

def normalize_text(s):
    '''Collapse whitespace so that two copies of the same item that differ only
    in incidental whitespace do not get different item IDs.'''
    return ' '.join(str(s).split())

def stable_hash(s) -> int:
    '''Deterministic replacement for Python's built-in hash(): gives the same
    value across processes and sessions, unlike hash() for strings, whose
    output depends on a per-process random seed unless PYTHONHASHSEED is
    pinned.'''
    return int(hashlib.sha256(str(s).encode('utf-8')).hexdigest(), 16) & 0xffffffff

def canonical_item_id_base(source, task, story, question) -> str:
    key = '|'.join([str(source), str(task), normalize_text(story), normalize_text(question)])
    return hashlib.sha256(key.encode('utf-8')).hexdigest()[:16]

def canonical_story_id(story) -> str:
    return hashlib.sha256(normalize_text(story).encode('utf-8')).hexdigest()[:16]

def prompt_hash_of(user_prompt) -> str:
    return hashlib.sha256(str(user_prompt).encode('utf-8')).hexdigest()[:16]

class _UniqueIdAssigner:
    '''If the same (source, task, story, question) combination legitimately
    appears more than once in the raw data, assign a deterministic -1, -2, ...
    suffix based on order of appearance instead of colliding on one item_id.'''
    def __init__(self):
        self._counts = {}
    def get(self, source, task, story, question):
        base = canonical_item_id_base(source, task, story, question)
        n = self._counts.get(base, 0)
        self._counts[base] = n + 1
        return base if n == 0 else f'{base}-{n}'

class ManifestIntegrityError(RuntimeError):
    pass

def assert_unique(df, col, where):
    dup = df[df.duplicated(col, keep=False)]
    if len(dup) > 0:
        raise ManifestIntegrityError(f'[{where}] {len(dup)} duplicate rows found in column {col}\\n{dup[[col]].head(10)}')

def assert_matched_join(left, right, on, where):
    '''Verify that two dataframes join 1:1 and completely on `on`. This is the
    minimum precondition before computing any paired statistic between them.'''
    l_ids, r_ids = set(left[on]), set(right[on])
    only_l, only_r = l_ids - r_ids, r_ids - l_ids
    if only_l or only_r:
        raise ManifestIntegrityError(
            f'[{where}] item_id mismatch: {len(only_l)} item(s) only on the left, '
            f'{len(only_r)} only on the right.\\n'
            f'These two result sets do not cover the same items -- refusing to compute a paired test.'
        )
    merged = left.merge(right, on=on, how='inner', validate='one_to_one', suffixes=('_L','_R'))
    assert len(merged) == len(left) == len(right)
    if 'prompt_hash_L' in merged.columns and 'prompt_hash_R' in merged.columns:
        mism = merged[merged['prompt_hash_L'] != merged['prompt_hash_R']]
        if len(mism) > 0:
            raise ManifestIntegrityError(
                f'[{where}] {len(mism)} row(s) share an item_id but have a different prompt_hash '
                f'-- same item, but the rendered prompt (e.g. option order) differs.'
            )
    print(f'  OK [{where}]: {len(merged)} rows matched 1:1, 0 item_id/prompt_hash mismatches')
    return merged

print('Integrity utilities ready (stable_hash, canonical_item_id_base, assert_matched_join, ...)')


## ToMBench loading and item-level split

In [ ]:

import glob, math, re, random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit

TOMBENCH_DATA_PATHS = [
    '/content/ToMBench/data', '/content/data',
    '/content/drive/MyDrive/ToMBench-main/data',
    '/content/drive/MyDrive/ToMBench/data',
]
OPENTOM_DATA_PATHS = [
    '/content/OpenToM/data',
    '/content/drive/MyDrive/OpenToM-main/data',
    '/content/drive/MyDrive/OpenToM/data',
]
TEST_RATIO = 0.30
VAL_RATIO_OF_TRAIN = 0.20
EVAL_MAX_PER_DATASET = 1000

TOMBENCH_DIR = None
for p in TOMBENCH_DATA_PATHS:
    if os.path.isdir(p) and glob.glob(os.path.join(p, '*.jsonl')):
        TOMBENCH_DIR = p; break
if TOMBENCH_DIR is None:
    subprocess.run(['git','clone','--depth','1','https://github.com/zhchen18/ToMBench.git','/content/ToMBench'], check=False)
    TOMBENCH_DIR = '/content/ToMBench/data'
assert os.path.isdir(TOMBENCH_DIR)
print(f'ToMBench data dir: {TOMBENCH_DIR}')

def is_valid(x):
    if x is None: return False
    if isinstance(x, float) and math.isnan(x): return False
    s = str(x).strip()
    return len(s) > 0 and s.lower() != 'nan'

def normalize_ability(ab):
    return ab.replace('Non-literal', 'Non-Literal')

def build_user_prompt(story, question, opts):
    lines = [f'{l}. {t}' for l, t in opts]
    return (f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(lines))

_tombench_ids = _UniqueIdAssigner()
records = []
for fp in sorted(glob.glob(os.path.join(TOMBENCH_DIR, '*.jsonl'))):
    task = os.path.splitext(os.path.basename(fp))[0]
    with open(fp, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = _json.loads(line)
            ability = normalize_ability(row.get('能力\nABILITY', 'Unknown'))
            category = ability.split(':')[0].strip()
            ans = row.get('答案\nANSWER') or row.get('ANSWER')
            if not is_valid(row.get('STORY')) or not is_valid(row.get('QUESTION')) or not is_valid(ans): continue
            opts = []
            for letter in ['A','B','C','D']:
                v = row.get(f'OPTION-{letter}')
                if is_valid(v): opts.append((letter, str(v).strip()))
            story, question = row['STORY'], row['QUESTION']
            up = build_user_prompt(story, question, opts)
            records.append({
                'source': 'ToMBench', 'task': task, 'category': category, 'ability': ability,
                'story': story, 'question': question,
                'user_prompt': up,
                'answer': str(ans).strip().upper()[0],
                'item_id': _tombench_ids.get('ToMBench', task, story, question),
                'prompt_hash': prompt_hash_of(up),
            })
df = pd.DataFrame(records)
assert_unique(df, 'item_id', 'ToMBench source data')
print(f'ToMBench: {len(df)} items / {df["ability"].nunique()} abilities / item_id uniqueness OK')

# --- Stratified split (ability-stratified for large groups; small groups get
#     a deterministic per-ability split so rare abilities are not dropped) ---
SMALL_THRESHOLD = 30
ab_counts = df['ability'].value_counts()
small_ab = ab_counts[ab_counts < SMALL_THRESHOLD].index.tolist()
large_ab = ab_counts[ab_counts >= SMALL_THRESHOLD].index.tolist()
large_df = df[df['ability'].isin(large_ab)].reset_index(drop=True)
train_val_l, test_l = train_test_split(large_df, test_size=TEST_RATIO, random_state=42, stratify=large_df['ability'])
train_l, val_l = train_test_split(train_val_l, test_size=VAL_RATIO_OF_TRAIN, random_state=42, stratify=train_val_l['ability'])
parts = {'train': [], 'val': [], 'test': []}
for ab in small_ab:
    ab_df = df[df['ability'] == ab].reset_index(drop=True)
    n = len(ab_df)
    if n < 5:
        parts['train'].append(ab_df); continue
    seed_ab = 42 + (stable_hash(ab) % 100)   # deterministic per-ability seed, not Python's hash()
    tr, vt = train_test_split(ab_df, test_size=0.4, random_state=seed_ab)
    va, te = train_test_split(vt, test_size=0.5, random_state=seed_ab)
    parts['train'].append(tr); parts['val'].append(va); parts['test'].append(te)
train_s = pd.concat(parts['train']) if parts['train'] else pd.DataFrame()
val_s = pd.concat(parts['val']) if parts['val'] else pd.DataFrame()
test_s = pd.concat(parts['test']) if parts['test'] else pd.DataFrame()
train_df = pd.concat([train_l, train_s], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = pd.concat([val_l, val_s],     ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = pd.concat([test_l, test_s],   ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}  (expected 1607/411/842)')
assert len(train_df)==1607 and len(val_df)==411 and len(test_df)==842, 'Split size does not match the expected counts -- check the source data version'

# Persist the split indexed by item_id -- the basis for every later join.
for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    out = split_df.assign(split=name)
    out[['item_id','source','task','category','ability','answer','split']].to_csv(f'{OUTPUT_DIR}/split_{name}_index_v2.csv', index=False)
    out.to_json(f'{OUTPUT_DIR}/split_{name}_full_v2.json', orient='records', force_ascii=False)
print('Wrote split_{train,val,test}_index_v2.csv / _full_v2.json (item_id included)')


## External benchmark loaders (OpenToM, ToMi, SocialIQa, Hi-ToM)

In [ ]:

def cap_eval(recs, n=EVAL_MAX_PER_DATASET):
    if n and len(recs) > n: return random.Random(42).sample(recs, n)
    return recs

# --- OpenToM ---
def load_opentom():
    ids = _UniqueIdAssigner()
    OPENTOM_DIR = None
    for p in OPENTOM_DATA_PATHS:
        if os.path.isdir(p): OPENTOM_DIR = p; break
    if OPENTOM_DIR is None:
        subprocess.run(['git','clone','--depth','1','https://github.com/seacowx/OpenToM.git','/content/OpenToM'], check=False)
        if os.path.isdir('/content/OpenToM/data'): OPENTOM_DIR = '/content/OpenToM/data'
    if OPENTOM_DIR is None: print('OpenToM data not found'); return []
    meta_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'meta_data.json')
    recs = []
    if os.path.exists(meta_path):
        with open(meta_path) as f: meta = _json.load(f)
        att_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'attitude.json')
        if os.path.exists(att_path):
            with open(att_path) as f: att = _json.load(f)
            opts_text = ['positive','negative','neutral']
            for sid, qs in att.items():
                narrative = meta.get(sid,{}).get('narrative','')
                if not narrative: continue
                for q in qs:
                    gold = q.get('answer','').strip().lower()
                    if gold not in opts_text: continue
                    shuffled = opts_text[:]
                    random.Random(stable_hash(sid+q.get('question',''))).shuffle(shuffled)
                    gold_letter = 'ABC'[shuffled.index(gold)]
                    up = build_user_prompt(narrative, q['question'], list(zip('ABC',shuffled)))
                    recs.append({
                        'source':'OpenToM','task':'attitude','category':'Emotion/Attitude','ability':'Attitude',
                        'story': narrative, 'question': q['question'],
                        'user_prompt': up, 'answer':gold_letter,
                        'item_id': ids.get('OpenToM','attitude',narrative,q['question']),
                        'prompt_hash': prompt_hash_of(up),
                    })
        cg_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'location_cg_fo.json')
        if os.path.exists(cg_path):
            with open(cg_path) as f: cg = _json.load(f)
            for sid, qs in cg.items():
                narrative = meta.get(sid,{}).get('narrative','')
                if not narrative: continue
                for q in qs:
                    gold = q.get('answer','').strip()
                    if gold not in ('Yes','No'): continue
                    opts = [('A','Yes'),('B','No')]
                    gold_letter = 'A' if gold == 'Yes' else 'B'
                    up = build_user_prompt(narrative, q['question'], opts)
                    recs.append({
                        'source':'OpenToM','task':'location-cg','category':'Belief','ability':'Location False Beliefs',
                        'story': narrative, 'question': q['question'],
                        'user_prompt': up, 'answer': gold_letter,
                        'item_id': ids.get('OpenToM','location-cg',narrative,q['question']),
                        'prompt_hash': prompt_hash_of(up),
                    })
    return cap_eval(recs)

# --- ToMi ---
def load_tomi():
    ids = _UniqueIdAssigner()
    TOMI_DIR = '/content/ToMi'
    if not os.path.isdir(TOMI_DIR):
        r = subprocess.run(['git','clone','--depth','1','https://github.com/facebookresearch/ToMi.git', TOMI_DIR], capture_output=True)
        if r.returncode != 0: print('ToMi clone failed'); return []
    DATA_OUT = '/content/tomi_data'
    os.makedirs(DATA_OUT, exist_ok=True)
    if not os.path.exists(os.path.join(DATA_OUT,'test.txt')):
        r = subprocess.run([sys.executable,'main.py','-n','500','-o',DATA_OUT,'-s','42'], cwd=TOMI_DIR, capture_output=True)
        if r.returncode != 0: print('ToMi generation failed'); return []
    samples = []; cur_facts, cur_qas = [], []; prev = 0
    with open(os.path.join(DATA_OUT,'test.txt')) as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip(): continue
            m = re.match(r'^(\d+)\s(.*)$', line)
            if not m: continue
            num = int(m.group(1)); rest = m.group(2)
            if num <= prev and (cur_facts or cur_qas):
                narr = ' '.join(cur_facts)
                for q,a in cur_qas: samples.append({'narrative':narr,'question':q,'answer':a})
                cur_facts, cur_qas = [], []
            prev = num
            if '\t' in rest:
                parts = rest.split('\t')
                if len(parts) >= 2: cur_qas.append((parts[0].strip(), parts[1].strip()))
            else:
                cur_facts.append(rest.strip())
        if cur_facts or cur_qas:
            narr = ' '.join(cur_facts)
            for q,a in cur_qas: samples.append({'narrative':narr,'question':q,'answer':a})
    ans_pool = list({s['answer'] for s in samples})
    if len(ans_pool) < 4: return []
    recs = []
    for s in samples:
        gold = s['answer']
        distractors = [a for a in ans_pool if a != gold]
        ds = random.Random(stable_hash(s['narrative'][:40])).sample(distractors, 3)
        opts = [gold] + ds
        random.Random(stable_hash(s['question'])).shuffle(opts)
        gold_letter = 'ABCD'[opts.index(gold)]
        up = build_user_prompt(s['narrative'], s['question'], list(zip('ABCD', opts)))
        recs.append({
            'source':'ToMi','task':'location-belief','category':'Belief','ability':'Location False Beliefs',
            'story': s['narrative'], 'question': s['question'],
            'user_prompt': up, 'answer': gold_letter,
            'item_id': ids.get('ToMi','location-belief',s['narrative'],s['question']),
            'prompt_hash': prompt_hash_of(up),
        })
    return cap_eval(recs)

# --- SocialIQa ---
def load_socialiqa():
    ids = _UniqueIdAssigner()
    from datasets import load_dataset
    ds = None
    for name in ['allenai/social_i_qa','social_i_qa']:
        try:
            ds = load_dataset(name, split='validation'); print(f'SocialIQa from HF: {name}'); break
        except Exception:
            try:
                ds = load_dataset(name, split='validation', trust_remote_code=True); print(f'SocialIQa from HF (legacy loader): {name}'); break
            except Exception as e2:
                print(f'  {name}: {type(e2).__name__}')
                continue
    if ds is None:
        try:
            import urllib.request, zipfile
            url = 'https://storage.googleapis.com/ai2-mosaic/public/socialiqa/socialiqa-train-dev.zip'
            zip_path = '/content/socialiqa.zip'
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path) as z: z.extractall('/content/socialiqa_data')
            dev_jsonl = glob.glob('/content/socialiqa_data/**/dev.jsonl', recursive=True)
            dev_lbl   = glob.glob('/content/socialiqa_data/**/dev-labels.lst', recursive=True)
            if dev_jsonl and dev_lbl:
                with open(dev_jsonl[0]) as f: rows = [_json.loads(l) for l in f]
                with open(dev_lbl[0]) as f: lbls = [l.strip() for l in f]
                ds = [dict(r, label=lbls[i]) for i,r in enumerate(rows)]
                print(f'SocialIQa from AllenAI zip: {len(ds)} items')
        except Exception as e:
            print(f'  AllenAI zip fallback failed: {e}')
    if ds is None: print('SocialIQa: all load paths failed, skipping'); return []
    recs = []
    for x in ds:
        ctx = x.get('context',''); q = x.get('question','')
        a,b,c = x.get('answerA',''), x.get('answerB',''), x.get('answerC','')
        lbl = str(x.get('label','')).strip()
        if lbl not in ('1','2','3'): continue
        gold_letter = {'1':'A','2':'B','3':'C'}[lbl]
        up = build_user_prompt(ctx, q, [('A',a),('B',b),('C',c)])
        recs.append({
            'source':'SocialIQa','task':'social_cs','category':'Mixed','ability':'Mixed',
            'story': ctx, 'question': q,
            'user_prompt': up, 'answer': gold_letter,
            'item_id': ids.get('SocialIQa','social_cs',ctx,q),
            'prompt_hash': prompt_hash_of(up),
        })
    return cap_eval(recs)

# --- Hi-ToM ---
def _parse_hitom_choices(choices_str):
    if not isinstance(choices_str, str): return []
    pat = re.compile(r'([A-Z])\.\s*([^,]+?)(?=,\s*[A-Z]\.|$)')
    return [(m.group(1).upper(), m.group(2).strip()) for m in pat.finditer(choices_str)]

def load_hitom():
    ids = _UniqueIdAssigner()
    from datasets import load_dataset
    ds = None
    for name in ['Hi-ToM/Hi-ToM_Dataset', 'umwyf/Hi-ToM_Dataset']:
        try:
            ds = load_dataset(name, split='train'); print(f'Hi-ToM from {name}'); break
        except Exception as e:
            print(f'  {name}: {type(e).__name__}')
    if ds is None:
        try:
            HITOM_DIR = '/content/Hi-ToM_dataset'
            if not os.path.isdir(HITOM_DIR):
                subprocess.run(['git','clone','--depth','1','https://github.com/ying-hui-he/Hi-ToM_dataset.git', HITOM_DIR], capture_output=True, check=False)
            json_files = glob.glob(f'{HITOM_DIR}/**/*.json', recursive=True) + glob.glob(f'{HITOM_DIR}/**/*.jsonl', recursive=True)
            if json_files:
                rows = []
                for fp in json_files:
                    with open(fp) as f:
                        try: data = _json.load(f); rows.extend(data if isinstance(data, list) else [data])
                        except: f.seek(0); rows.extend(_json.loads(l) for l in f if l.strip())
                if rows: ds = rows; print(f'Hi-ToM from GitHub clone: {len(rows)} items')
        except Exception as e:
            print(f'  GitHub clone fallback failed: {e}')
    if ds is None: print('Hi-ToM: all load paths failed, skipping'); return []

    recs = []
    skipped = 0
    LETTERS = 'ABCDEFGHIJKLMNO'
    for x in ds:
        story = x.get('story') or x.get('context') or x.get('narrative') or ''
        q = x.get('question') or ''
        choices_raw = x.get('choices') or x.get('options') or []
        gold_raw = x.get('answer')
        if gold_raw is None: gold_raw = x.get('label')
        if not story or not q or not choices_raw or gold_raw is None: skipped += 1; continue

        if isinstance(choices_raw, str):
            opts = _parse_hitom_choices(choices_raw)
            if not opts:
                try:
                    parsed = _json.loads(choices_raw)
                    opts = list(zip(LETTERS, parsed))
                except:
                    parts = [p.strip() for p in choices_raw.split('|')]
                    opts = list(zip(LETTERS, parts))
        elif isinstance(choices_raw, list):
            opts = list(zip(LETTERS, choices_raw))
        else:
            skipped += 1; continue
        if not opts: skipped += 1; continue

        gold_str = str(gold_raw).strip()
        gold_letter = None
        if len(gold_str) == 1 and gold_str.upper() in LETTERS:
            gold_letter = gold_str.upper()
        else:
            for letter, text in opts:
                if text.lower() == gold_str.lower():
                    gold_letter = letter; break
            if gold_letter is None:
                for letter, text in opts:
                    if gold_str.lower() in text.lower() or text.lower() in gold_str.lower():
                        gold_letter = letter; break
        if gold_letter is None: skipped += 1; continue

        if len(opts) > 5:
            gold_opt = next(o for o in opts if o[0] == gold_letter)
            others = [o for o in opts if o[0] != gold_letter]
            sampled = random.Random(stable_hash(story+q)).sample(others, min(4, len(others)))
            new_pool = [gold_opt] + sampled
            random.Random(stable_hash(q)).shuffle(new_pool)
            opts = [('ABCDE'[i], t) for i,(_, t) in enumerate(new_pool)]
            for i,(_, t) in enumerate(new_pool):
                if t == gold_opt[1]: gold_letter = 'ABCDE'[i]; break

        order = x.get('question_order', None)
        ability_label = f'High-Order False Beliefs (order={order})' if order is not None else 'High-Order False Beliefs'
        up = build_user_prompt(story, q, opts)
        recs.append({
            'source':'HiToM','task':'high-order','category':'Belief','ability':ability_label,
            'story': story, 'question': q,
            'user_prompt': up, 'answer': gold_letter,
            'item_id': ids.get('HiToM','high-order',story,q),
            'prompt_hash': prompt_hash_of(up),
            'gold_position': gold_letter,           # used for the gold-position confusion analysis
            'n_options': len(opts),
        })
    if skipped: print(f'  Hi-ToM: skipped {skipped} item(s) (answer/option matching failed)')
    return cap_eval(recs)

print('Loaders ready (all use stable_hash; every record carries item_id/story/question)')


## External evaluation manifests: build once, then reuse

In [ ]:

# Re-running this cell is safe: if a manifest is already saved, it is loaded
# as-is rather than regenerated, so that every downstream condition (multi-seed
# fine-tuning, few-shot conditions, etc.) is guaranteed to see the identical
# item set, run after run.

def _manifest_path(name):
    return f'{OUTPUT_DIR}/manifest_{name}.json'

def build_or_load_manifest(name, loader_fn):
    path = _manifest_path(name)
    if os.path.exists(path):
        recs = _json.load(open(path))
        print(f'[loaded] {name}: {len(recs)} items from {path}')
        return recs
    recs = loader_fn()
    with open(path, 'w') as f:
        _json.dump(recs, f, ensure_ascii=False, indent=2)
    print(f'[built] {name}: {len(recs)} items, saved to {path}')
    return recs

eval_sets = {'ToMBench': test_df.to_dict('records')}

_loaders = {'OpenToM': load_opentom, 'ToMi': load_tomi, 'SocialIQa': load_socialiqa, 'HiToM': load_hitom}
for name, fn in _loaders.items():
    recs = build_or_load_manifest(name, fn)
    if recs: eval_sets[name] = recs

eval_sets = {k:v for k,v in eval_sets.items() if len(v) > 0}
print(f'\nDatasets to evaluate: {list(eval_sets.keys())}')

# Integrity check: item_id must be unique within each dataset's manifest.
for name, recs in eval_sets.items():
    d = pd.DataFrame(recs)
    assert_unique(d, 'item_id', f'{name} manifest')
print('All manifests verified unique -- every condition below reuses these manifests as-is')


## Model and evaluation helpers

In [ ]:

from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)

ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')
def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt, few_shot_prefix=None):
    msgs = [{'role':'system','content':SYSTEM_PROMPT}]
    if few_shot_prefix:
        msgs += few_shot_prefix
    msgs.append({'role':'user','content':user_prompt})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def to_sft_text(tokenizer, user_prompt, gold):
    msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':user_prompt},
            {'role':'assistant','content':f'[[{gold}]]'}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

@torch.no_grad()
def evaluate(eval_records, model, tokenizer, desc='Eval', few_shot_prefix=None, max_new_tokens=12):
    # max_new_tokens=12 is enough for conditions trained/prompted to emit
    # '[[X]]' immediately (base, fine-tuned, answer-only, synthetic-format-only).
    # Conditions that write a reasoning sentence before the answer (rationale-CoT)
    # need a larger budget, or the generation is cut off before the [[X]] marker
    # appears and falls back to a default prediction -- callers of this function
    # pass a larger max_new_tokens for that condition.
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    results = []
    for r in tqdm(eval_records, desc=desc):
        prompt = to_text(tokenizer, r['user_prompt'], few_shot_prefix=few_shot_prefix)
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = extract_letter(gen)
        results.append({
            'item_id': r['item_id'], 'prompt_hash': r['prompt_hash'],
            'story_id': canonical_story_id(r['story']),   # for story-cluster-aware statistics
            'source': r['source'], 'task': r['task'], 'category': r['category'], 'ability': r['ability'],
            'answer': r['answer'], 'pred': pred, 'raw': gen, 'correct': int(pred == r['answer']),
            'n_options': r.get('n_options'),   # Hi-ToM option count (NaN for other datasets)
            'marker_found': '[[' in gen,   # False means the [[X]] marker was not reached within max_new_tokens
        })
    return pd.DataFrame(results)

def load_fresh_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
    return model, tokenizer

def save_results(df, dataset_name, condition, seed=None, exemplar_seed=None):
    tag = condition + (f'_seed{seed}' if seed is not None else '') + (f'_ex{exemplar_seed}' if exemplar_seed is not None else '')
    df = df.assign(condition=condition, seed=seed, exemplar_seed=exemplar_seed)
    path = f'{OUTPUT_DIR}/{tag}_{dataset_name}.csv'
    df.to_csv(path, index=False)
    return path

print('Model/evaluation helpers ready (item_id always included in output)')


## Baseline evaluation (zero-shot, fixed manifest)

In [ ]:

print('='*60); print('Baseline evaluation'); print('='*60)
print('   (already-saved baseline_*.csv files are loaded as-is on rerun, without recomputing)')

baseline_dfs = {}
_todo = {n: r for n, r in eval_sets.items() if not os.path.exists(f'{OUTPUT_DIR}/baseline_{n}.csv')}
if _todo:
    base_model, base_tokenizer = load_fresh_model()
else:
    base_model, base_tokenizer = None, None

for name, recs in eval_sets.items():
    path = f'{OUTPUT_DIR}/baseline_{name}.csv'
    if os.path.exists(path):
        df_r = pd.read_csv(path)
        baseline_dfs[name] = df_r
        print(f'  [skip] {name}: already done (loaded from disk) -- acc={df_r["correct"].mean()*100:.2f}% (n={len(df_r)})')
        continue
    df_r = evaluate(recs, base_model, base_tokenizer, desc=f'Baseline {name}')
    baseline_dfs[name] = df_r
    save_results(df_r, name, 'baseline')
    print(f'  {name}: acc={df_r["correct"].mean()*100:.2f}%  (n={len(df_r)})')

if base_model is not None:
    del base_model, base_tokenizer
    torch.cuda.empty_cache()
print('Baseline evaluation complete')


## Multi-seed QLoRA fine-tuning and evaluation

In [ ]:

from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported

SEEDS = [42, 43, 44, 45, 46]   # reduce this list from the end if resources are limited
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2

def train_and_eval_seed(seed):
    print('='*60); print(f'Training seed {seed}'); print('='*60)
    model, tokenizer = load_fresh_model()
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=seed,
    )
    _gpu = torch.cuda.get_device_name(0)
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if 'A100' in _gpu or _vram >= 35: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 8, 1, 16
    elif 'L4' in _gpu or _vram >= 22: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 4, 2, 8
    else: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 2, 4, 4

    train_texts = [to_sft_text(tokenizer, r['user_prompt'], r['answer']) for r in train_df.to_dict('records')]
    val_texts   = [to_sft_text(tokenizer, r['user_prompt'], r['answer']) for r in val_df.to_dict('records')]
    train_ds_text = Dataset.from_dict({'text': train_texts})
    val_ds_text   = Dataset.from_dict({'text': val_texts})

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds_text, eval_dataset=val_ds_text,
        dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
            gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=0.03, num_train_epochs=NUM_EPOCHS,
            learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=20, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
            metric_for_best_model='eval_loss', greater_is_better=False, save_total_limit=2,
            optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=seed,
            output_dir=f'outputs_seed{seed}', report_to='none',
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    t0 = time.time()
    trainer.train()
    train_time = time.time()-t0
    print(f'Seed {seed} training complete ({train_time/60:.1f} min)')

    # ToMBench in-domain test (item-level split) plus all four external benchmarks,
    # all through the same evaluate() function and the same fixed manifests.
    in_domain = evaluate(test_df.to_dict('records'), model, tokenizer, desc=f'seed{seed}-ToMBench')
    save_results(in_domain, 'ToMBench', 'finetuned', seed=seed)
    accs = {'ToMBench': in_domain['correct'].mean()*100}
    for name, recs in eval_sets.items():
        if name == 'ToMBench': continue
        res_df = evaluate(recs, model, tokenizer, desc=f'seed{seed}-{name}')
        save_results(res_df, name, 'finetuned', seed=seed)
        accs[name] = res_df['correct'].mean()*100

    if seed == SEEDS[0]:
        model.save_pretrained(f'{OUTPUT_DIR}/adapter_seed{seed}')
        tokenizer.save_pretrained(f'{OUTPUT_DIR}/adapter_seed{seed}')

    del model, tokenizer, trainer
    torch.cuda.empty_cache()
    accs['_train_time_sec'] = train_time
    return accs

def _seed_output_paths(seed):
    return [f'{OUTPUT_DIR}/finetuned_seed{seed}_{n}.csv' for n in eval_sets.keys()]

def _seed_already_done(seed):
    return all(os.path.exists(p) for p in _seed_output_paths(seed))

def _load_seed_results(seed):
    accs = {}
    for n in eval_sets.keys():
        df_r = pd.read_csv(f'{OUTPUT_DIR}/finetuned_seed{seed}_{n}.csv')
        accs[n] = df_r['correct'].mean() * 100
    accs['_train_time_sec'] = None
    return accs

print('   (seeds with all output files already present are loaded from disk, skipping retraining)')
multi_seed_results = {}
for s in SEEDS:
    if _seed_already_done(s):
        print(f'[skip] seed {s}: already done (loaded from disk)')
        multi_seed_results[s] = _load_seed_results(s)
    else:
        multi_seed_results[s] = train_and_eval_seed(s)

print('\nPer-seed results:'); print(_json.dumps(multi_seed_results, indent=2, ensure_ascii=False))
with open(f'{OUTPUT_DIR}/multi_seed_results_v2.json','w') as f:
    _json.dump(multi_seed_results, f, indent=2, ensure_ascii=False)

# Baseline is seed-independent (the base model is never trained) -- reuse baseline_dfs.
rows = []
for ds in eval_sets.keys():
    base_acc = baseline_dfs[ds]['correct'].mean()*100
    deltas = [multi_seed_results[s][ds] - base_acc for s in SEEDS if ds in multi_seed_results.get(s, {})]
    rows.append({'dataset': ds, 'baseline_acc': base_acc, 'n_seeds': len(deltas),
                 'delta_mean': float(np.mean(deltas)), 'delta_std': float(np.std(deltas, ddof=1)) if len(deltas)>1 else 0.0,
                 'deltas_per_seed': deltas})
multi_seed_summary = pd.DataFrame(rows)
print(multi_seed_summary.to_string(index=False))
multi_seed_summary.to_csv(f'{OUTPUT_DIR}/multi_seed_summary_v2.csv', index=False)
print('Wrote multi_seed_summary_v2.csv -- every number here comes from the shared manifest, no hardcoded values')


## Inference performance: latency, throughput, peak VRAM

Measures single-item inference latency, throughput, and peak GPU memory for the
base model and the seed-42 adapter on the same GPU used for training, to
support the deployability discussion with measured figures. **Report the GPU
name printed below** -- Colab assigns a different GPU per session, so the
actual hardware used should be cited alongside any latency/throughput number.

In [ ]:

print('='*60); print('Inference performance measurement'); print('='*60)

import statistics

def measure_latency(model, tokenizer, records, n=100, warmup=5):
    FastLanguageModel.for_inference(model)
    sample = records[:n] if len(records) <= n else [records[i] for i in
              sorted(random.Random(42).sample(range(len(records)), n))]
    # Warm up first (excludes CUDA kernel caching / first-call overhead from the measurement).
    for r in sample[:warmup]:
        prompt = to_text(tokenizer, r['user_prompt'])
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    lat_ms = []
    t_start = time.time()
    for r in sample:
        prompt = to_text(tokenizer, r['user_prompt'])
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        torch.cuda.synchronize(); t0 = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        lat_ms.append((time.time()-t0)*1000)
    total_s = time.time() - t_start
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
    lat_ms.sort()
    return {
        'n': len(sample),
        'latency_ms_mean': statistics.mean(lat_ms),
        'latency_ms_median': statistics.median(lat_ms),
        'latency_ms_p95': lat_ms[int(0.95*len(lat_ms))-1],
        'throughput_items_per_sec': len(sample) / total_s,
        'peak_vram_gb': peak_vram_gb,
    }

_perf_path = f'{OUTPUT_DIR}/inference_perf_v2.json'
if os.path.exists(_perf_path):
    print(f'[skip] already done (loaded from disk): {_perf_path}')
    perf_results = _json.load(open(_perf_path))
    print(perf_results)
else:
    perf_records = test_df.to_dict('records')  # ToMBench test set, same items for base and adapter

    perf_results = {}
    gpu_name_for_report = torch.cuda.get_device_name(0)
    vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

    print('--- Base model (no adapter) ---')
    pm, pt = load_fresh_model()
    perf_results['base'] = measure_latency(pm, pt, perf_records)
    print(perf_results['base'])
    del pm, pt; torch.cuda.empty_cache()

    print('--- Fine-tuned (seed42 adapter) ---')
    from peft import PeftModel
    pm2, pt2 = load_fresh_model()
    pm2 = PeftModel.from_pretrained(pm2, f'{OUTPUT_DIR}/adapter_seed42')
    perf_results['finetuned_seed42'] = measure_latency(pm2, pt2, perf_records)
    print(perf_results['finetuned_seed42'])
    del pm2, pt2; torch.cuda.empty_cache()

    perf_results['_meta'] = {'gpu_name': gpu_name_for_report, 'gpu_vram_total_gb': vram_total_gb, 'measured_on': 'ToMBench item-level test set (n=842, sampled 100)'}
    with open(_perf_path, 'w') as f:
        _json.dump(perf_results, f, indent=2, ensure_ascii=False)
    print(f"\nWrote inference_perf_v2.json -- cite as 'measured on {gpu_name_for_report} ({vram_total_gb:.0f}GB)'.")
    print('   If a specific target deployment device exists (e.g. an on-device chip), it should be measured separately -- these figures are for the Colab GPU used here.')


## Story-level group split (story-disjoint control, with a 3-way overlap check)

In [ ]:

print('='*60); print('Story-level group split'); print('='*60)

def extract_story(user_prompt):
    m = re.search(r'\[Story\]\n(.*?)\n\n\[Question\]', user_prompt, re.DOTALL)
    return m.group(1) if m else user_prompt[:200]

df_g = df.copy()
df_g['canonical_story_id'] = df_g['story'].apply(canonical_story_id)   # grouping key is story identity alone, not (task, story)
groups = df_g['canonical_story_id'].values

gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=42)
trainval_idx, test_idx = next(gss1.split(df_g, groups=groups))
trainval_df_g = df_g.iloc[trainval_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_RATIO_OF_TRAIN, random_state=42)
train_idx2, val_idx2 = next(gss2.split(trainval_df_g, groups=trainval_df_g['canonical_story_id'].values))
train_df_g = trainval_df_g.iloc[train_idx2].reset_index(drop=True)
val_df_g   = trainval_df_g.iloc[val_idx2].reset_index(drop=True)
test_df_g  = df_g.iloc[test_idx].reset_index(drop=True)

print(f'Group split sizes: Train={len(train_df_g)} Val={len(val_df_g)} Test={len(test_df_g)}')

# Check all three pairs (train-val, train-test, val-test) for story overlap.
pairs = [('train','val',train_df_g,val_df_g), ('train','test',train_df_g,test_df_g), ('val','test',val_df_g,test_df_g)]
for a_name, b_name, a_df, b_df in pairs:
    overlap = set(a_df['canonical_story_id']) & set(b_df['canonical_story_id'])
    print(f'  {a_name} vs {b_name} story overlap: {len(overlap)} (expected 0)')
    assert len(overlap) == 0, f'Story leakage remains between {a_name} and {b_name}'
print('All three pairs confirmed to have zero story overlap')

_gs_base_path = f'{OUTPUT_DIR}/groupsplit_base_ToMBench.csv'
_gs_tuned_path = f'{OUTPUT_DIR}/groupsplit_finetuned_ToMBench.csv'
if os.path.exists(_gs_base_path) and os.path.exists(_gs_tuned_path):
    print('[skip] group-split training/eval already done (loaded from disk), skipping retraining')
    base_res_g = pd.read_csv(_gs_base_path)
    tuned_res_g = pd.read_csv(_gs_tuned_path)
    base_acc_g = base_res_g['correct'].mean()*100
    tuned_acc_g = tuned_res_g['correct'].mean()*100
else:
    g_model, g_tokenizer = load_fresh_model()
    g_model = FastLanguageModel.get_peft_model(
        g_model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )
    _gpu = torch.cuda.get_device_name(0)
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if 'A100' in _gpu or _vram >= 35: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 8, 1, 16
    elif 'L4' in _gpu or _vram >= 22: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 4, 2, 8
    else: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 2, 4, 4

    train_texts_g = [to_sft_text(g_tokenizer, r['user_prompt'], r['answer']) for r in train_df_g.to_dict('records')]
    val_texts_g   = [to_sft_text(g_tokenizer, r['user_prompt'], r['answer']) for r in val_df_g.to_dict('records')]
    train_ds_g = Dataset.from_dict({'text': train_texts_g})
    val_ds_g   = Dataset.from_dict({'text': val_texts_g})

    g_trainer = SFTTrainer(
        model=g_model, tokenizer=g_tokenizer, train_dataset=train_ds_g, eval_dataset=val_ds_g,
        dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
            gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=0.03, num_train_epochs=NUM_EPOCHS,
            learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=20, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
            metric_for_best_model='eval_loss', greater_is_better=False, save_total_limit=2,
            optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=42,
            output_dir='outputs_groupsplit_v2', report_to='none',
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    g_trainer.train()

    base_model_g, base_tok_g = load_fresh_model()
    base_res_g = evaluate(test_df_g.to_dict('records'), base_model_g, base_tok_g, desc='group-test-base')
    base_acc_g = base_res_g['correct'].mean()*100
    del base_model_g, base_tok_g
    torch.cuda.empty_cache()

    tuned_res_g = evaluate(test_df_g.to_dict('records'), g_model, g_tokenizer, desc='group-test-tuned')
    tuned_acc_g = tuned_res_g['correct'].mean()*100

    save_results(base_res_g, 'ToMBench', 'groupsplit_base')
    save_results(tuned_res_g, 'ToMBench', 'groupsplit_finetuned')

    del g_model, g_tokenizer, g_trainer
    torch.cuda.empty_cache()

# Compare against the item-level split result (fixed manifest, seed 42) for reference.
item_level_base_acc = baseline_dfs['ToMBench']['correct'].mean()*100
item_level_tuned_acc = multi_seed_results[42]['ToMBench']
print(f'\n[Group split, no story leakage in test] base={base_acc_g:.2f}%  tuned={tuned_acc_g:.2f}%  delta={tuned_acc_g-base_acc_g:+.2f}pp')
print(f'[Item-level split, seed42]              base={item_level_base_acc:.2f}%  tuned={item_level_tuned_acc:.2f}%  delta={item_level_tuned_acc-item_level_base_acc:+.2f}pp')

with open(f'{OUTPUT_DIR}/groupsplit_summary_v2.json','w') as f:
    _json.dump({'base_acc': base_acc_g, 'tuned_acc': tuned_acc_g, 'delta': tuned_acc_g-base_acc_g,
               'n_train': len(train_df_g), 'n_val': len(val_df_g), 'n_test': len(test_df_g),
               'item_level_delta_seed42': item_level_tuned_acc-item_level_base_acc}, f, indent=2)

print('Group-split results saved')


## Training-free few-shot conditions: synthetic-format-only, answer-only, rationale-CoT

In [ ]:

print('='*60); print('Few-shot conditions'); print('='*60)

EXEMPLAR_SEEDS = [0, 1, 2]   # three independent exemplar draws/orderings per condition

def pick_exemplars(exemplar_seed, k=3):
    idx = train_df.sample(n=k, random_state=100+exemplar_seed).index.tolist()
    return train_df.loc[idx]

def build_rationale_cot_fewshot(exemplar_seed):
    '''3-shot exemplars that include a brief rationale before the [[X]] answer.'''
    exemplars = pick_exemplars(exemplar_seed)
    msgs = []
    for _, r in exemplars.iterrows():
        reasoning = (f"Let's reason step by step: I track who has access to which information in the story, "
                     f"then check what the relevant character would infer given only what they observed. "
                     f"This points to option {r['answer']}.")
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"{reasoning} [[{r['answer']}]]"})
    return msgs

def build_answer_only_3shot(exemplar_seed):
    '''3-shot in-context learning: full task content (story/question/options)
    plus the gold answer, no reasoning. Content-bearing, so this is a task
    demonstration, not a content-free format probe.'''
    exemplars = pick_exemplars(exemplar_seed)
    msgs = []
    for _, r in exemplars.iterrows():
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"[[{r['answer']}]]"})
    return msgs

_SYNTH_TOPICS = [
    ('[Story]\nA red box and a blue box sit on a table.\n\n[Question]\nWhich box is red?\n\n[Candidate Answers]\nA. The red box\nB. The blue box', 'A'),
    ('[Story]\nThere are three cups: one has water, one has juice, one is empty.\n\n[Question]\nWhich cup is empty?\n\n[Candidate Answers]\nA. The water cup\nB. The juice cup\nC. The empty cup', 'C'),
    ('[Story]\nA number sequence: 2, 4, 6, 8.\n\n[Question]\nWhat comes next?\n\n[Candidate Answers]\nA. 9\nB. 10\nC. 12', 'B'),
    ('[Story]\nA clock shows 3:00. One hour passes.\n\n[Question]\nWhat time is it now?\n\n[Candidate Answers]\nA. 2:00\nB. 4:00\nC. 5:00', 'B'),
    ('[Story]\nA shelf has 5 books. 2 are removed.\n\n[Question]\nHow many books remain?\n\n[Candidate Answers]\nA. 2\nB. 3\nC. 5', 'B'),
]

def build_synthetic_format_only(exemplar_seed):
    '''Content-free control: toy questions unrelated to ToMBench, exposing only
    the [[X]] output format and no task content.'''
    rng = random.Random(200+exemplar_seed)
    picks = rng.sample(_SYNTH_TOPICS, 3)
    msgs = []
    for prompt, ans in picks:
        msgs.append({'role':'user','content': prompt})
        msgs.append({'role':'assistant','content': f'[[{ans}]]'})
    return msgs

CONDITIONS = {
    'rationale_cot_3shot': build_rationale_cot_fewshot,
    'answer_only_3shot': build_answer_only_3shot,
    'synthetic_format_only_3shot': build_synthetic_format_only,
}

print('   (already-saved files are skipped on rerun; rationale_cot_3shot additionally re-patches any row still flagged as truncated by a prior run)')
print('   (rationale_cot_3shot patches only the rows still flagged by marker_found=False AND a long raw generation -- not a full rerun)')
fs_model, fs_tokenizer = None, None  # loaded lazily, only if something actually needs computing

fewshot_all_results = {}
for cond_name, builder in CONDITIONS.items():
    for ex_seed in EXEMPLAR_SEEDS:
        tag = f'{cond_name}_ex{ex_seed}'
        names_needed = list(eval_sets.keys())
        all_exist = all(os.path.exists(f'{OUTPUT_DIR}/{tag}_{n}.csv') for n in names_needed)
        # answer_only/synthetic_format_only reach the [[X]] marker essentially every
        # time (checked empirically), so once all their files exist they are
        # skipped outright. rationale_cot_3shot always falls through to the
        # per-dataset loop below, since an existing file may still contain rows
        # that were truncated by an earlier, shorter generation-length budget.
        if all_exist and cond_name != 'rationale_cot_3shot':
            print(f'[skip] [{cond_name} ex{ex_seed}] all datasets already done')
            for n in names_needed:
                df_r = pd.read_csv(f'{OUTPUT_DIR}/{tag}_{n}.csv')
                acc = df_r['correct'].mean()*100
                fewshot_all_results.setdefault(cond_name, {}).setdefault(n, []).append(acc)
            continue
        prefix = None  # built lazily, only once this exemplar seed actually needs generation
        for name, recs in eval_sets.items():
            # rationale-CoT writes a reasoning sentence before the answer, so a
            # 12-token budget (enough for the other two conditions, whose output
            # is a few characters) is not enough and needs to be raised.
            _max_new = 500 if cond_name == 'rationale_cot_3shot' else 12
            out_path = f'{OUTPUT_DIR}/{tag}_{name}.csv'
            if os.path.exists(out_path):
                res_df = pd.read_csv(out_path)
                n_bad = 0
                if cond_name == 'rationale_cot_3shot' and 'marker_found' in res_df.columns:
                    # marker_found=False alone is not a reliable truncation signal: most
                    # such rows are short, complete answers in a slightly different format
                    # (e.g. "[B]" or "B. some_option" instead of "[[B]]"), already correctly
                    # scored by the fallback extraction rule. Genuinely truncated rows are
                    # long (a partial reasoning sentence, typically 250+ characters), so
                    # requiring both conditions avoids reprocessing already-fine rows.
                    bad_mask = (~res_df['marker_found'].astype(bool)) & (res_df['raw'].astype(str).str.len() > 50)
                    n_bad = int(bad_mask.sum())
                if n_bad == 0:
                    print(f'  [skip] [{cond_name} ex{ex_seed}] {name}: already done, no rows need patching')
                else:
                    if fs_model is None:
                        fs_model, fs_tokenizer = load_fresh_model()
                    if prefix is None:
                        prefix = builder(ex_seed)
                    bad_ids = set(res_df.loc[bad_mask, 'item_id'])
                    recs_redo = [r for r in recs if r['item_id'] in bad_ids]
                    print(f'  [patch] [{cond_name} ex{ex_seed}] {name}: regenerating {n_bad}/{len(res_df)} flagged row(s) at {_max_new} tokens')
                    redo_df = evaluate(recs_redo, fs_model, fs_tokenizer, desc=f'{cond_name}-ex{ex_seed}-{name}-fix', few_shot_prefix=prefix, max_new_tokens=_max_new)
                    res_df = pd.concat([res_df[~bad_mask], redo_df], ignore_index=True)
                    save_results(res_df, name, cond_name, exemplar_seed=ex_seed)
                    print(f'  [done] [{cond_name} ex{ex_seed}] {name}: patched and saved')
            else:
                if fs_model is None:
                    fs_model, fs_tokenizer = load_fresh_model()
                if prefix is None:
                    prefix = builder(ex_seed)
                res_df = evaluate(recs, fs_model, fs_tokenizer, desc=f'{cond_name}-ex{ex_seed}-{name}', few_shot_prefix=prefix, max_new_tokens=_max_new)
                save_results(res_df, name, cond_name, exemplar_seed=ex_seed)
            acc = res_df['correct'].mean()*100
            fewshot_all_results.setdefault(cond_name, {}).setdefault(name, []).append(acc)
            print(f'  [{cond_name} ex{ex_seed}] {name}: acc={acc:.2f}%')

if fs_model is not None:
    del fs_model, fs_tokenizer
    torch.cuda.empty_cache()

summary_rows = []
for cond_name, per_ds in fewshot_all_results.items():
    for ds, accs in per_ds.items():
        summary_rows.append({'condition': cond_name, 'dataset': ds, 'n_exemplar_seeds': len(accs),
                              'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs, ddof=1)) if len(accs)>1 else 0.0,
                              'accs': accs})
fewshot_summary = pd.DataFrame(summary_rows)
print(fewshot_summary.to_string(index=False))
fewshot_summary.to_csv(f'{OUTPUT_DIR}/fewshot_conditions_summary_v2.csv', index=False)
print('Wrote fewshot_conditions_summary_v2.csv')


## Final integrity check: verify every output is item-identity-consistent

In [ ]:

print('='*60); print('Final integrity check across all outputs'); print('='*60)

def _load(tag, ds):
    p = f'{OUTPUT_DIR}/{tag}_{ds}.csv'
    return pd.read_csv(p) if os.path.exists(p) else None

seed_tags = [f'finetuned_seed{s}' for s in SEEDS]
fewshot_tags = [f'{cond}_ex{ex}' for cond in CONDITIONS.keys() for ex in EXEMPLAR_SEEDS]

for ds in eval_sets.keys():
    base = _load('baseline', ds)
    if base is None: continue
    for tag in seed_tags + fewshot_tags:
        other = _load(tag, ds)
        if other is None: continue
        assert_matched_join(base[['item_id','prompt_hash']], other[['item_id','prompt_hash']], 'item_id', f'{ds}: baseline vs {tag}')

print(f'\nAll checks passed: baseline vs. all {len(seed_tags)} seeds vs. all {len(fewshot_tags)} few-shot conditions '
      'have identical item_id/prompt_hash sets.')
print('   Paired tests (McNemar, etc.) can now be safely computed on these outputs. '
      '(The group split is intentionally excluded from this check, since it uses a deliberately different test set.)')


## Notes on data availability

This study fine-tunes on 70% of ToMBench, whose original documentation describes it as an
evaluation-only benchmark. The manuscript (Section 5.5) discloses this departure explicitly,
including the rationale and limitations, rather than treating the benchmark's evaluation-only
guidance as waived.

The full `tier4_v2_fixed/` output folder (per-item predictions, manifests, and the
gold-answer-position analysis CSVs) and a version-pinned `requirements.lock.txt` are released
alongside this notebook in the repository linked from the manuscript's Data Availability
Statement.

---
## Story-disjoint (group-split) model, evaluated on all four external benchmarks

The original run trained a story-disjoint model (`GroupShuffleSplit` on `canonical_story_id`,
same r=16/seed-42 recipe) but only evaluated it in-domain on `test_df_g`, then discarded the
adapter. That model's cross-benchmark behavior was never checked, so RQ2's "central
contribution" rested entirely on a model trained under the item-level split's ~60% story
overlap between train and test.

This cell reproduces that exact split and training recipe, but additionally (a) saves the
adapter, and (b) evaluates the resulting model on `eval_sets['OpenToM']`, `eval_sets['ToMi']`,
`eval_sets['SocialIQa']`, `eval_sets['HiToM']` -- the identical external manifests used for the
seed-42 primary comparison, so the results are directly comparable to `Table 4`/`Table 5` in
the manuscript. The external benchmarks were never part of ToMBench's train/test split, so
there is no leakage concern for them specifically; what changes is that the underlying model
was fine-tuned without any story memorization shortcut on the ToMBench side.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported

NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2

print('Rebuilding story-level group split')
df_g = df.copy()
df_g['canonical_story_id'] = df_g['story'].apply(canonical_story_id)
groups = df_g['canonical_story_id'].values

gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=42)
trainval_idx, test_idx = next(gss1.split(df_g, groups=groups))
trainval_df_g = df_g.iloc[trainval_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_RATIO_OF_TRAIN, random_state=42)
train_idx2, val_idx2 = next(gss2.split(trainval_df_g, groups=trainval_df_g['canonical_story_id'].values))
train_df_g = trainval_df_g.iloc[train_idx2].reset_index(drop=True)
val_df_g   = trainval_df_g.iloc[val_idx2].reset_index(drop=True)
test_df_g  = df_g.iloc[test_idx].reset_index(drop=True)
print(f'Group split sizes: Train={len(train_df_g)} Val={len(val_df_g)} Test={len(test_df_g)} (expect ~1647/376/837)')

for a_name, b_name, a_df, b_df in [('train','val',train_df_g,val_df_g), ('train','test',train_df_g,test_df_g), ('val','test',val_df_g,test_df_g)]:
    overlap = set(a_df['canonical_story_id']) & set(b_df['canonical_story_id'])
    assert len(overlap) == 0, f'Story leakage remains between {a_name} and {b_name}'
print('All three pairs confirmed zero story overlap')

ADAPTER_GROUPSPLIT_DIR = f'{OUTPUT_DIR}/adapter_groupsplit'
_m2_ext_names = [n for n in eval_sets.keys() if n != 'ToMBench']
_m2_done = os.path.isdir(ADAPTER_GROUPSPLIT_DIR) and all(
    os.path.exists(f'{OUTPUT_DIR}/groupsplit_finetuned_{n}.csv') for n in eval_sets.keys()
)

if _m2_done:
    print('[skip] group-split model already complete (adapter + all result files present) -- loading from disk')
    m2_results = {n: pd.read_csv(f'{OUTPUT_DIR}/groupsplit_finetuned_{n}.csv') for n in eval_sets.keys()}
else:
    print('Training group-split model (r=16, seed=42, 4 epochs, early stopping)...')
    g_model, g_tokenizer = load_fresh_model()
    g_model = FastLanguageModel.get_peft_model(
        g_model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )
    TRAIN_BS, GRAD_ACCUM, EVAL_BS = gpu_batch_sizes()
    train_texts_g = [to_sft_text(g_tokenizer, r['user_prompt'], r['answer']) for r in train_df_g.to_dict('records')]
    val_texts_g   = [to_sft_text(g_tokenizer, r['user_prompt'], r['answer']) for r in val_df_g.to_dict('records')]
    train_ds_g = Dataset.from_dict({'text': train_texts_g})
    val_ds_g   = Dataset.from_dict({'text': val_texts_g})

    g_trainer = SFTTrainer(
        model=g_model, tokenizer=g_tokenizer, train_dataset=train_ds_g, eval_dataset=val_ds_g,
        dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
            gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=0.03, num_train_epochs=NUM_EPOCHS,
            learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=20, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
            metric_for_best_model='eval_loss', greater_is_better=False, save_total_limit=2,
            optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=42,
            output_dir='outputs_groupsplit_m2', report_to='none',
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    t0 = time.time()
    g_trainer.train()
    print(f'Training complete ({(time.time()-t0)/60:.1f} min)')

    g_model.save_pretrained(ADAPTER_GROUPSPLIT_DIR)
    g_tokenizer.save_pretrained(ADAPTER_GROUPSPLIT_DIR)
    print(f'Saved adapter to {ADAPTER_GROUPSPLIT_DIR}')

    m2_results = {}
    in_domain_g = evaluate(test_df_g.to_dict('records'), g_model, g_tokenizer, desc='GroupSplit-ToMBench(group-test)')
    save_results(in_domain_g, 'ToMBench', 'groupsplit_finetuned')
    m2_results['ToMBench'] = in_domain_g

    for name in _m2_ext_names:
        res_df = evaluate(eval_sets[name], g_model, g_tokenizer, desc=f'GroupSplit-{name}')
        save_results(res_df, name, 'groupsplit_finetuned')
        m2_results[name] = res_df

    del g_model, g_tokenizer, g_trainer
    torch.cuda.empty_cache()

print('\nGroup-split results (group-split-trained model vs. baseline, same external manifests as Table 4):')
for name, res_df in m2_results.items():
    tuned_acc = res_df['correct'].mean() * 100
    if name == 'ToMBench':
        base_ref = evaluate(test_df_g.to_dict('records'), *load_fresh_model(), desc='GroupSplit-ToMBench-base-ref') if False else None
        print(f'  ToMBench (group-test, n={len(res_df)}): tuned_acc={tuned_acc:.2f}% '
              f'(compare to groupsplit_summary_v2.json for the base figure already on record)')
    else:
        base_acc = baseline_dfs[name]['correct'].mean() * 100
        print(f'  {name} (n={len(res_df)}): base={base_acc:.2f}%  group-split-tuned={tuned_acc:.2f}%  '
              f'delta={tuned_acc-base_acc:+.2f}pp  (item-level-split seed42 delta for reference: see Table 4)')
print('\nWrote groupsplit_finetuned_{ToMBench,OpenToM,ToMi,SocialIQa,HiToM}.csv and adapter_groupsplit/.')
print('Bring these files back to the local analysis environment and re-run analyze_tier4_v2.py')
print('-- it will pick up groupsplit_finetuned_<dataset>.csv for any dataset beyond ToMBench automatically')
print('once matched against baseline_<dataset>.csv with the same join logic used elsewhere in that script.')

---
## Fine-tuned (seed-42) model + the same three training-free 3-shot conditions

Table 12 in the manuscript currently reports the three few-shot conditions (answer-only,
synthetic-format-only, rationale-CoT) applied only to the zero-shot **base** model. This
section applies the identical three conditions, exemplar seeds, and decoding settings to the
already **fine-tuned** (seed-42) model, completing the 2x4 design (base/fine-tuned x
zero-shot/answer-only/format-only/rationale-CoT) for a symmetric comparison with the base-model
few-shot results. Exemplars are drawn from
`train_df` exactly as in the original few-shot cell, using the same `pick_exemplars` seeding
scheme, so the same three exemplar draws (ex0-ex2) are reused.

In [ ]:
print('Loading fine-tuned (seed-42) adapter for few-shot conditions')
from peft import PeftModel

_m6_base, _m6_tok = load_fresh_model()
m6_model = PeftModel.from_pretrained(_m6_base, f'{OUTPUT_DIR}/adapter_seed42')
m6_tokenizer = _m6_tok
print('Fine-tuned (seed-42) adapter loaded on top of the base 4-bit model.')

EXEMPLAR_SEEDS = [0, 1, 2]

def pick_exemplars(exemplar_seed, k=3):
    idx = train_df.sample(n=k, random_state=100+exemplar_seed).index.tolist()
    return train_df.loc[idx]

def build_rationale_cot_fewshot(exemplar_seed):
    exemplars = pick_exemplars(exemplar_seed)
    msgs = []
    for _, r in exemplars.iterrows():
        reasoning = (f"Let's reason step by step: I track who has access to which information in the story, "
                     f"then check what the relevant character would infer given only what they observed. "
                     f"This points to option {r['answer']}.")
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"{reasoning} [[{r['answer']}]]"})
    return msgs

def build_answer_only_3shot(exemplar_seed):
    exemplars = pick_exemplars(exemplar_seed)
    msgs = []
    for _, r in exemplars.iterrows():
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"[[{r['answer']}]]"})
    return msgs

_SYNTH_TOPICS = [
    ('[Story]\nA red box and a blue box sit on a table.\n\n[Question]\nWhich box is red?\n\n[Candidate Answers]\nA. The red box\nB. The blue box', 'A'),
    ('[Story]\nThere are three cups: one has water, one has juice, one is empty.\n\n[Question]\nWhich cup is empty?\n\n[Candidate Answers]\nA. The water cup\nB. The juice cup\nC. The empty cup', 'C'),
    ('[Story]\nA number sequence: 2, 4, 6, 8.\n\n[Question]\nWhat comes next?\n\n[Candidate Answers]\nA. 9\nB. 10\nC. 12', 'B'),
    ('[Story]\nA clock shows 3:00. One hour passes.\n\n[Question]\nWhat time is it now?\n\n[Candidate Answers]\nA. 2:00\nB. 4:00\nC. 5:00', 'B'),
    ('[Story]\nA shelf has 5 books. 2 are removed.\n\n[Question]\nHow many books remain?\n\n[Candidate Answers]\nA. 2\nB. 3\nC. 5', 'B'),
]

def build_synthetic_format_only(exemplar_seed):
    rng = random.Random(200+exemplar_seed)
    picks = rng.sample(_SYNTH_TOPICS, 3)
    msgs = []
    for prompt, ans in picks:
        msgs.append({'role':'user','content': prompt})
        msgs.append({'role':'assistant','content': f'[[{ans}]]'})
    return msgs

CONDITIONS = {
    'rationale_cot_3shot': build_rationale_cot_fewshot,
    'answer_only_3shot': build_answer_only_3shot,
    'synthetic_format_only_3shot': build_synthetic_format_only,
}

m6_results = {}
for cond_name, builder in CONDITIONS.items():
    tuned_cond = f'{cond_name}_tuned'
    for ex_seed in EXEMPLAR_SEEDS:
        tag = f'{tuned_cond}_ex{ex_seed}'
        prefix = None
        for name, recs in eval_sets.items():
            out_path = f'{OUTPUT_DIR}/{tag}_{name}.csv'
            if os.path.exists(out_path):
                res_df = pd.read_csv(out_path)
                print(f'  [skip] [{tuned_cond} ex{ex_seed}] {name}: already done')
            else:
                if prefix is None:
                    prefix = builder(ex_seed)
                _max_new = 500 if cond_name == 'rationale_cot_3shot' else 12
                res_df = evaluate(recs, m6_model, m6_tokenizer, desc=f'{tuned_cond}-ex{ex_seed}-{name}',
                                   few_shot_prefix=prefix, max_new_tokens=_max_new)
                save_results(res_df, name, tuned_cond, exemplar_seed=ex_seed)
            acc = res_df['correct'].mean()*100
            m6_results.setdefault(cond_name, {}).setdefault(name, []).append(acc)
            print(f'  [{tuned_cond} ex{ex_seed}] {name}: acc={acc:.2f}%')

del m6_model, m6_tokenizer, _m6_base, _m6_tok
torch.cuda.empty_cache()

print('\nSummary (fine-tuned model + 3-shot, mean over 3 exemplar seeds), for comparison against')
print('Table 12 (base model + 3-shot) and Table 11 (fine-tuned zero-shot, seed 42 primary run):')
m6_rows = []
for cond_name, per_ds in m6_results.items():
    for ds, accs in per_ds.items():
        m6_rows.append({'condition': f'{cond_name}_tuned', 'dataset': ds, 'n_exemplar_seeds': len(accs),
                         'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs, ddof=1)) if len(accs)>1 else 0.0})
m6_summary = pd.DataFrame(m6_rows)
print(m6_summary.to_string(index=False))
m6_summary.to_csv(f'{OUTPUT_DIR}/fewshot_tuned_conditions_summary_v2.csv', index=False)
print('\nWrote {condition}_tuned_ex{0,1,2}_{dataset}.csv and fewshot_tuned_conditions_summary_v2.csv')

---
## fp16 (non-quantized) LoRA control

Every fine-tuning run reported in the manuscript starts from the same 4-bit pre-quantized
checkpoint (`unsloth/Qwen2.5-3B-Instruct-bnb-4bit`), so effects attributed to LoRA adaptation
are not separated from any interaction between quantization and adaptation. This section
trains the same LoRA recipe (r=16, alpha=16, seed 42) on top of the standard, non-quantized
`unsloth/Qwen2.5-3B-Instruct` checkpoint in fp16/bf16, and evaluates both the fp16 base model
and the fp16-LoRA-tuned model on all five benchmarks, so the resulting deltas can be compared
directly against the 4-bit deltas in Table 4.

**VRAM note:** fp16 weights for a 3B model are roughly 4x the 4-bit footprint (~6GB just for
weights). This comfortably fits a T4 (16GB) or better; `gpu_batch_sizes()` already scales batch
size down for smaller GPUs, but if you hit an out-of-memory error, reduce
`per_device_train_batch_size` further and increase `gradient_accumulation_steps` to compensate,
or switch the Colab runtime to a larger GPU.

In [ ]:
print('fp16 (non-quantized) baseline + LoRA control')
FP16_MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct'   # standard checkpoint, NOT the -bnb-4bit variant

# --- fp16 base (zero-shot) evaluation, all five datasets ---
_p3_base_done = all(os.path.exists(f'{OUTPUT_DIR}/baseline_fp16_{n}.csv') for n in eval_sets.keys())
if _p3_base_done:
    print('[skip] fp16 baseline already complete -- loading from disk')
    baseline_fp16_dfs = {n: pd.read_csv(f'{OUTPUT_DIR}/baseline_fp16_{n}.csv') for n in eval_sets.keys()}
else:
    fp16_base_model, fp16_base_tok = load_fresh_model(model_name=FP16_MODEL_NAME, load_in_4bit=False)
    baseline_fp16_dfs = {}
    for name, recs in eval_sets.items():
        res_df = evaluate(recs, fp16_base_model, fp16_base_tok, desc=f'fp16-base-{name}')
        save_results(res_df, name, 'baseline_fp16')
        baseline_fp16_dfs[name] = res_df
        print(f'  fp16 base {name}: acc={res_df["correct"].mean()*100:.2f}%')
    del fp16_base_model, fp16_base_tok
    torch.cuda.empty_cache()

# --- fp16 LoRA fine-tuning (r=16, seed=42), all five datasets ---
ADAPTER_FP16_DIR = f'{OUTPUT_DIR}/adapter_fp16_seed42'
_p3_tuned_done = os.path.isdir(ADAPTER_FP16_DIR) and all(
    os.path.exists(f'{OUTPUT_DIR}/finetuned_fp16_seed42_{n}.csv') for n in eval_sets.keys()
)
if _p3_tuned_done:
    print('[skip] fp16 fine-tuned model already complete -- loading from disk')
    finetuned_fp16_dfs = {n: pd.read_csv(f'{OUTPUT_DIR}/finetuned_fp16_seed42_{n}.csv') for n in eval_sets.keys()}
else:
    fp16_model, fp16_tok = load_fresh_model(model_name=FP16_MODEL_NAME, load_in_4bit=False)
    fp16_model = FastLanguageModel.get_peft_model(
        fp16_model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )
    TRAIN_BS, GRAD_ACCUM, EVAL_BS = gpu_batch_sizes()
    train_texts_fp16 = [to_sft_text(fp16_tok, r['user_prompt'], r['answer']) for r in train_df.to_dict('records')]
    val_texts_fp16   = [to_sft_text(fp16_tok, r['user_prompt'], r['answer']) for r in val_df.to_dict('records')]
    train_ds_fp16 = Dataset.from_dict({'text': train_texts_fp16})
    val_ds_fp16   = Dataset.from_dict({'text': val_texts_fp16})

    fp16_trainer = SFTTrainer(
        model=fp16_model, tokenizer=fp16_tok, train_dataset=train_ds_fp16, eval_dataset=val_ds_fp16,
        dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
            gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=0.03, num_train_epochs=NUM_EPOCHS,
            learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=20, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
            metric_for_best_model='eval_loss', greater_is_better=False, save_total_limit=2,
            optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=42,
            output_dir='outputs_fp16_p3', report_to='none',
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    t0 = time.time()
    fp16_trainer.train()
    print(f'fp16 training complete ({(time.time()-t0)/60:.1f} min)')

    fp16_model.save_pretrained(ADAPTER_FP16_DIR)
    fp16_tok.save_pretrained(ADAPTER_FP16_DIR)
    print(f'Saved adapter to {ADAPTER_FP16_DIR}')

    finetuned_fp16_dfs = {}
    for name, recs in eval_sets.items():
        res_df = evaluate(recs, fp16_model, fp16_tok, desc=f'fp16-tuned-{name}')
        save_results(res_df, name, 'finetuned_fp16', seed=42)
        finetuned_fp16_dfs[name] = res_df
        print(f'  fp16 tuned {name}: acc={res_df["correct"].mean()*100:.2f}%')

    del fp16_model, fp16_tok, fp16_trainer
    torch.cuda.empty_cache()

print('\nSummary (fp16, non-quantized LoRA control) vs. the manuscript\'s 4-bit results:')
p3_rows = []
for name in eval_sets.keys():
    base_acc = baseline_fp16_dfs[name]['correct'].mean() * 100
    tuned_acc = finetuned_fp16_dfs[name]['correct'].mean() * 100
    delta = tuned_acc - base_acc
    p3_rows.append({'dataset': name, 'fp16_base_acc': base_acc, 'fp16_tuned_acc': tuned_acc, 'fp16_delta_pp': delta})
    print(f'  {name}: fp16 base={base_acc:.2f}%  fp16 tuned={tuned_acc:.2f}%  delta={delta:+.2f}pp  '
          f'(compare to the 4-bit seed-42 delta for the same dataset in Table 4)')
pd.DataFrame(p3_rows).to_csv(f'{OUTPUT_DIR}/ANALYSIS_fp16_control_summary_v2.csv', index=False)
print('\nWrote baseline_fp16_<dataset>.csv, finetuned_fp16_seed42_<dataset>.csv, adapter_fp16_seed42/,')
print('and ANALYSIS_fp16_control_summary_v2.csv. McNemar/bootstrap-CI significance testing on these')
print('(matching Table 4/5 format) should be run locally with analyze_tier4_v2.py after downloading.')

---
## Summary and next steps

1. Confirm all three sections above completed without error (check the printed per-dataset
   accuracies against the sanity ranges already reported for the 4-bit seed-42 run in the
   manuscript -- large unexplained deviations, e.g. >10pp on a dataset that should be similar,
   are worth double-checking before trusting the numbers).
2. Download (or sync via Drive) the full `tier4_v2_fixed/` folder, which now additionally
   contains: `groupsplit_finetuned_{OpenToM,ToMi,SocialIQa,HiToM}.csv` and `adapter_groupsplit/`;
   `{condition}_tuned_ex{0,1,2}_{dataset}.csv` and `fewshot_tuned_conditions_summary_v2.csv`;
   `baseline_fp16_*.csv`, `finetuned_fp16_seed42_*.csv`, `adapter_fp16_seed42/`, and
   `ANALYSIS_fp16_control_summary_v2.csv`.
3. Re-run `code/analyze_tier4_v2.py --dir tier4_v2_fixed` locally (or extend it, as was done for
   the position-bias and residual-effect checks already in the manuscript) to compute McNemar
   tests and story-cluster bootstrap CIs on the new result pairs.
4. Bring the resulting numbers back into `main.tex` (Data Availability Statement, Limitations,
   and the corresponding results discussion) if not already finalized there.

In [ ]:
print('Done. See the "Summary and next steps" cell above for what to do with these outputs.')